# BindCraft Binder Design — One-Off Sampling Example

![BindCraft Binder Design — One-Off Sampling Example](https://proto-bio.github.io/proto-assets/images/tool/bindcraft/hero.png)

Design a single protein binder against a target with the BindCraft pipeline
(AlphaFold2 hallucination + ProteinMPNN refinement + PyRosetta filtering).

- Paper: [Pacesa et al., Nature 2025](https://www.nature.com/articles/s41586-025-09429-6)
- Upstream repo (pinned commit `7cd4ace`): https://github.com/martinpacesa/BindCraft/tree/7cd4ace1b7407adf66a50dfefa47de2270f5e4a9

This example uses tiny iteration counts and `max_trajectories=1` for a
~10-30 minute run on an H100. For production runs (100 binders, 24-72h),
leave `BindCraftConfig` at its upstream defaults and set
`number_of_final_designs=100` on the input.

> **Note: long-running tool.** Binder design campaigns take a long time on GPU, so the design cells below are left unexecuted and shown for reference.

In [ ]:
from proto_tools.tools.binder_design.bindcraft import (
    BindCraftConfig,
    BindCraftInput,
    run_bindcraft_design,
)
from proto_tools.utils.notebook_docs import display_api_reference


## Input API Reference

In [ ]:
display_api_reference("bindcraft", "input")

## Config API Reference

In [ ]:
display_api_reference("bindcraft", "config")

## Output API Reference

In [ ]:
display_api_reference("bindcraft", "output")

In [ ]:
from pathlib import Path

# Resolve the in-tree renin test target by walking up to the repo root.
_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
pdb_path = _repo_root / "tests" / "dummy_data" / "renin_af3.pdb"
print(f"Using target PDB: {pdb_path}")

inputs = BindCraftInput(
    target_pdb=str(pdb_path),
    target_chain="A",
    target_hotspot_residues="56",          # 1-indexed residue position(s) on the target
    binder_lengths=(60, 70),               # (min, max)
    binder_name="example",
    number_of_final_designs=1,             # one-off sampling
)

config = BindCraftConfig(
    max_trajectories=1,                    # cap at 1 trajectory
    soft_iterations=10,                    # tiny iteration budget
    temporary_iterations=5,
    hard_iterations=2,
    greedy_iterations=2,
    num_seqs=2,
    max_mpnn_sequences=1,
    seed=42,                               # reproducible
)

result = run_bindcraft_design(inputs, config)
print(f"Trajectories run: {result.n_trajectories_run}")
print(f"Designs accepted: {result.n_designs_accepted}")

In [ ]:
if result.designs:
    design = result.designs[0]
    print(f"Design: {design.design_name}")
    print(f"Sequence ({len(design.binder_sequence)} aa): {design.binder_sequence}")
    print(f"Seed: {design.seed}")
    print(f"avg pLDDT: {design.metrics['avg_plddt']:.3f}")
    print(f"avg iPTM:  {design.metrics['avg_iptm']:.3f}")
    print(f"dG:        {design.metrics['dG']:.2f} REU")
    print(f"Shape complementarity: {design.metrics['shape_complementarity']:.3f}")
else:
    print("No designs passed the filter check — try more trajectories or relaxed filters.")

## Tuning runtime

BindCraft is a stochastic, filter-driven pipeline — most trajectories fail the filter check, so the wall-clock time depends on both the number of accepted designs you want and how strict the filters are. Three useful operating points:

- **One-off sampling (~10-30 min on H100)**: this notebook — `number_of_final_designs=1`, `max_trajectories=1`, tiny iteration counts. Good for smoke-testing the pipeline end-to-end on a new target.
- **Quick scan (~2-4h)**: `number_of_final_designs=5`, `max_trajectories=50`, leave iteration counts at upstream defaults. Enough designs to see if the target is tractable at all.
- **Production run (~24-72h)**: `number_of_final_designs=100`, no `max_trajectories` cap, default iteration counts. Matches the upstream paper's per-target budget.

In [ ]:
# Demonstrate `filter_overrides`: tighten Average_pLDDT and Average_i_pTM beyond
# the defaults. This cell only constructs the config (no GPU / no subprocess) so
# the notebook can be executed end-to-end within reasonable time budgets — to
# actually run, pass `strict_inputs` + `strict_config` to `run_bindcraft_design`.
strict_inputs = BindCraftInput(
    target_pdb=str(pdb_path),
    target_chain="A",
    target_hotspot_residues="56",
    binder_lengths=(60, 70),
    binder_name="strict",
    number_of_final_designs=1,
)
strict_config = BindCraftConfig(
    max_trajectories=1,
    soft_iterations=10, temporary_iterations=5,
    hard_iterations=2, greedy_iterations=2,
    num_seqs=2, max_mpnn_sequences=1,
    seed=42,
    filter_overrides={
        "Average_pLDDT": {"threshold": 0.85, "higher": True},
        "Average_i_pTM": {"threshold": 0.6, "higher": True},
    },
)

print("Strict config:")
print(f"  filter_overrides = {strict_config.filter_overrides}")
# To actually run: strict_result = run_bindcraft_design(strict_inputs, strict_config)

In [ ]:
if result.designs:
    result.export("my_binder", file_format="pdb")  # writes a directory of PDBs + stats.json